In [1]:
!pip install torch -q
!pip install torch_geometric -q

In [2]:
import pandas as pd
import torch
import pickle 

from torch import nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Setting a Baseline - Just do Logistic Regression

Just use the data below and then create separate KNN models for each target variable, which are:
1. Fielder-Type (binary where a character can only be an on or off fielder)
2. DPS (binary)
3. Support (binary)
4. Survivability (binary)

In [4]:
df_characteristics = pd.read_csv('data/characters/characteristics.csv')

In [6]:
df_characteristics.head()

,Name,Element,On-Field,Off-Field,DPS,Support,Survivability,Weapon Type,Region,Body Type,Line Count,Star,Use,Own,Pull Number,Free
0,Aino,Hydro,0,1,1,0,0,Claymore,Nodkrai,Loli,430,4,22134,76710,283366,0
1,Albedo,Geo,0,1,1,0,0,Sword,Mondstadt,Boy,1434,5,84,8433,29861,0
2,Alhaitham,Dendro,1,0,1,0,0,Sword,Sumeru,Male,1226,5,2394,26184,56138,0
3,Aloy,Cryo,1,0,1,0,0,Bow,Ranger,Girl,0,5,12,19206,19206,0
4,Amber,Pyro,0,1,1,0,0,Bow,Mondstadt,Girl,779,4,75,77430,250873,0


In [5]:
# === 0. Imports ===
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from torch_geometric.utils import from_networkx

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

# === 1. Load your data ===
df = df_characteristics.copy()

# Columns
cat_cols = ['Element', 'Weapon Type', 'Region', 'Body Type']
binary_cols = ['On-Field', 'DPS', 'Support', 'Survivability', 'Star', 'Free']
num_cols = ['Line Count']
target_col = 'Pull Number'

# === 2. Preprocess categorical features: map to indices ===
cat_maps = {}
cat_indices = []

for col in cat_cols:
    categories = df[col].unique()
    cat_map = {v: i for i, v in enumerate(categories)}
    cat_maps[col] = cat_map
    cat_indices.append(df[col].map(cat_map).values)

cat_indices = torch.tensor(list(zip(*cat_indices)), dtype=torch.long, device=device)

# === 3. Binary features ===
binary_feats = torch.tensor(df[binary_cols].values, dtype=torch.float32, device=device)

# === 4. Numeric features ===
scaler = StandardScaler()
num_feats = torch.tensor(scaler.fit_transform(df[num_cols]), dtype=torch.float32, device=device)

# === 5. Node-level train/test split ===
num_nodes = len(df)
all_indices = list(range(num_nodes))
train_idx, test_idx = train_test_split(all_indices, test_size=0.2, random_state=42)
train_idx = torch.tensor(train_idx, dtype=torch.long, device=device)
test_idx = torch.tensor(test_idx, dtype=torch.long, device=device)

# === 6. NetworkX graph ===
# Replace with your real NetworkX graph
# Example: fully connected graph
with open("data/abyss_graphs/graph_52.pickle", "rb") as f:
    G_nx = pickle.load(f)

# Convert to PyG Data object
data = from_networkx(G_nx).to(device)
data = from_networkx(G_nx).to(device)

# === 7. GAT model ===
class GenshinGAT(nn.Module):
    def __init__(self, cat_dims, embedding_dims, num_bin, num_num, hidden_dim=16):
        super().__init__()
        # Embeddings for categorical features
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_categories, emb_dim)
            for num_categories, emb_dim in zip(cat_dims, embedding_dims)
        ])
        input_dim = sum(embedding_dims) + num_bin + num_num
        
        # GAT layers
        self.gat1 = GATConv(input_dim, hidden_dim, heads=2, concat=True)
        self.gat2 = GATConv(hidden_dim * 2, 1, heads=1, concat=False)  # regression output

    def forward(self, cat_idx, binary_feats, numeric_feats, edge_index):
        x_cat = torch.cat([emb(cat_idx[:, i]) for i, emb in enumerate(self.embeddings)], dim=1)
        x = torch.cat([x_cat, binary_feats, numeric_feats], dim=1)
        x = F.relu(self.gat1(x, edge_index))
        x = self.gat2(x, edge_index)
        return x.squeeze()  # shape: (num_nodes,)

# === 8. Instantiate model ===
embedding_dims = [4, 5, 5, 4]  # example sizes
cat_dims = [len(cat_maps[col]) for col in cat_cols]
num_bin = len(binary_cols)
num_num = len(num_cols)

model = GenshinGAT(cat_dims, embedding_dims, num_bin, num_num).to(device)

# Target
target = torch.tensor(df[target_col].values, dtype=torch.float32, device=device)

# === 9. Loss and optimizer ===
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

# === 10. Training loop ===
model.train()
for epoch in range(200):
    optimizer.zero_grad()
    out = model(cat_indices, binary_feats, num_feats, data.edge_index)
    
    # Only compute loss on training nodes
    loss = loss_fn(out[train_idx], target[train_idx])
    
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 20 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.2f}")

# === 11. Evaluation ===
model.eval()
with torch.no_grad():
    out = model(cat_indices, binary_feats, num_feats, data.edge_index)
    test_preds = out[test_idx]
    test_target = target[test_idx]
    
    mse = nn.MSELoss()(test_preds, test_target).item()
    rmse = mse ** 0.5
    print(f"\nTest RMSE: {rmse:.2f}")

    # Optional: simple MAPE
    mape = (torch.abs((test_preds - test_target) / (test_target + 1e-6))).mean().item() * 100
    print(f"Test MAPE: {mape:.2f}%")


Using cpu device
Epoch 20, Loss: 91079254016.00
Epoch 40, Loss: 91038793728.00
Epoch 60, Loss: 90947354624.00
Epoch 80, Loss: 90767163392.00
Epoch 100, Loss: 90447224832.00
Epoch 120, Loss: 89923141632.00
Epoch 140, Loss: 89164210176.00
Epoch 160, Loss: 88133779456.00
Epoch 180, Loss: 86794108928.00
Epoch 200, Loss: 85115551744.00

Test RMSE: 274345.03
Test MAPE: 87.99%


In [7]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression

from utils import ForecastingMetrics as FM

## Preprocessing and Train/Test Split

In [10]:
target_map = {
    "Fielder-Type": "On-Field",
    "DPS": "DPS",
    "Support": "Support",
    "Survivability": "Survivability",
}

categorical_cols = ["Element", "Weapon Type", "Region", "Body Type"]

results = {}

# Split config
kfold = KFold(n_splits=5, shuffle=True, random_state=37)

## Hyperparameter tuning for kNN

In [20]:
tuned_models = {}

for model_name, target_col in target_map.items():
    # 1. Build X and y
    X = df.drop(columns=["Name", target_col])
    y = df[target_col].values

    # Identify numeric columns
    numeric_cols = [c for c in X.columns if c not in categorical_cols]

    # 2. Train-test split (fixed for consistency)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=37
    )

    # 3. Pipeline Preprocessing
    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
            ("num", StandardScaler(), numeric_cols),
        ]
    )

    # 4. KNN Pipeline
    knn_pipe = Pipeline([
        ("prep", preprocessor),
        ("knn", KNeighborsRegressor())
    ])

    # 5. Hyperparameter Search Space
    param_grid = {
        "knn__n_neighbors": [3, 5, 7, 9, 11],
        "knn__weights": ["uniform", "distance"]
    }

    tuned_model = GridSearchCV(
        knn_pipe,
        param_grid=param_grid,
        scoring="neg_mean_squared_error",
        cv=kfold,
        n_jobs=-1,
        refit=True
    )

    tuned_model.fit(X_train, y_train)

    print(f"Best params for {model_name}: {tuned_model.best_params_}")

    tuned_models[model_name] = {
        "best_model": tuned_model.best_estimator_,
        "X_test": X_test,
        "y_test": y_test
    }

Best params for Fielder-Type: {'knn__n_neighbors': 3, 'knn__weights': 'distance'}
Best params for DPS: {'knn__n_neighbors': 3, 'knn__weights': 'uniform'}
Best params for Support: {'knn__n_neighbors': 3, 'knn__weights': 'uniform'}
Best params for Survivability: {'knn__n_neighbors': 5, 'knn__weights': 'uniform'}


In [19]:
linear_models = {}

for model_name, target_col in target_map.items():
    # Rebuild X/y to stay consistent
    X = df.drop(columns=["Name", target_col])
    y = df[target_col].values

    numeric_cols = [c for c in X.columns if c not in categorical_cols]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=37
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
            ("num", StandardScaler(), numeric_cols),
        ]
    )

    lr_pipe = Pipeline([
        ("prep", preprocessor),
        ("lr", LinearRegression())
    ])

    lr_pipe.fit(X_train, y_train)
    preds = lr_pipe.predict(X_test)

    linear_models[model_name] = {
        "model": lr_pipe,
        "y_true": y_test,
        "y_pred": preds
    }

In [17]:
for model_name, bundle in tuned_models.items():
    model = bundle["best_model"]
    X_test = bundle["X_test"]
    y_test = bundle["y_test"]

    preds = model.predict(X_test)

    tuned_models[model_name]["y_pred"] = preds
    tuned_models[model_name]["metrics"] = FM.compute_all_metrics(y_test, preds)

    print("Metrics:", tuned_models[model_name]["metrics"])

Metrics: {'MAE': 0.10377227, 'MSE': 0.042697046, 'RMSE': 0.20663263, 'MAPE': 7.4313865, 'SMAPE': 72.497665, 'Directional Accuracy': 90.0}
Metrics: {'MAE': 0.25396827, 'MSE': 0.15873016, 'RMSE': 0.39840955, 'MAPE': 23.076923, 'SMAPE': 90.52631, 'Directional Accuracy': 70.0}
Metrics: {'MAE': 0.2857143, 'MSE': 0.1904762, 'RMSE': 0.4364358, 'MAPE': 61.904762, 'SMAPE': 138.18182, 'Directional Accuracy': 90.0}
Metrics: {'MAE': 0.2952381, 'MSE': 0.18095239, 'RMSE': 0.42538497, 'MAPE': 32.5, 'SMAPE': 118.46561, 'Directional Accuracy': 90.0}


In [15]:
final_results = {}

for model_name in target_map.keys():

    final_results[model_name] = {
        "KNN": tuned_models[model_name]["metrics"],
        "Linear Regression": FM.compute_all_metrics(
            linear_models[model_name]["y_true"],
            linear_models[model_name]["y_pred"]
        )
    }

final_results

{'Fielder-Type': {'KNN': {'MAE': 0.10377227,
   'MSE': 0.042697046,
   'RMSE': 0.20663263,
   'MAPE': 7.4313865,
   'SMAPE': 72.497665,
   'Directional Accuracy': 90.0},
  'Linear Regression': {'MAE': 0.053584564,
   'MSE': 0.0046509597,
   'RMSE': 0.06819794,
   'MAPE': 4.632787,
   'SMAPE': 125.63605,
   'Directional Accuracy': 80.0}},
 'DPS': {'KNN': {'MAE': 0.25396827,
   'MSE': 0.15873016,
   'RMSE': 0.39840955,
   'MAPE': 23.076923,
   'SMAPE': 90.52631,
   'Directional Accuracy': 70.0},
  'Linear Regression': {'MAE': 0.2918475,
   'MSE': 0.15608256,
   'RMSE': 0.39507285,
   'MAPE': 25.469515,
   'SMAPE': 99.587036,
   'Directional Accuracy': 80.0}},
 'Support': {'KNN': {'MAE': 0.2857143,
   'MSE': 0.1904762,
   'RMSE': 0.4364358,
   'MAPE': 61.904762,
   'SMAPE': 138.18182,
   'Directional Accuracy': 90.0},
  'Linear Regression': {'MAE': 0.32797584,
   'MSE': 0.19523858,
   'RMSE': 0.44185808,
   'MAPE': 52.47888,
   'SMAPE': 160.78506,
   'Directional Accuracy': 65.0}},
 'Surv

In [16]:
rows = []

for target, models in final_results.items():
    for model_name, metrics in models.items():
        row = {"Target": target, "Model": model_name}
        row.update(metrics)  # add metric values
        rows.append(row)

results_df = pd.DataFrame(rows)

results_df

,Target,Model,MAE,MSE,RMSE,MAPE,SMAPE,Directional Accuracy
0,Fielder-Type,KNN,0.103772,0.042697,0.206633,7.431386,72.497665,90.0
1,Fielder-Type,Linear Regression,0.053585,0.004651,0.068198,4.632787,125.636047,80.0
2,DPS,KNN,0.253968,0.158730,0.398410,23.076923,90.526314,70.0
3,DPS,Linear Regression,0.291847,0.156083,0.395073,25.469515,99.587036,80.0
4,Support,KNN,0.285714,0.190476,0.436436,61.904762,138.181824,90.0
5,Support,Linear Regression,0.327976,0.195239,0.441858,52.478882,160.785065,65.0
6,Survivability,KNN,0.295238,0.180952,0.425385,32.500000,118.465607,90.0
7,Survivability,Linear Regression,0.362303,0.189519,0.435338,42.226624,147.108994,75.0
